In [0]:
%pip install google-ads
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS googleads_bronze
""")

DataFrame[]

In [0]:
spark.catalog.listDatabases()

[Database(name='bronze', catalog='workspace', description='', locationUri=''),
 Database(name='cognifyz', catalog='workspace', description='', locationUri=''),
 Database(name='dbt_rshaik', catalog='workspace', description='', locationUri=''),
 Database(name='default', catalog='workspace', description='Default schema (auto-created)', locationUri=''),
 Database(name='gold', catalog='workspace', description='', locationUri=''),
 Database(name='googleads_analytics', catalog='workspace', description='', locationUri=''),
 Database(name='googleads_bronze', catalog='workspace', description='', locationUri=''),
 Database(name='googleads_gold', catalog='workspace', description='', locationUri=''),
 Database(name='googleads_silver', catalog='workspace', description='', locationUri=''),
 Database(name='information_schema', catalog='workspace', description='Information schema (auto-created)', locationUri=''),
 Database(name='raw', catalog='workspace', description='', locationUri=''),
 Database(name

In [0]:
from google.ads.googleads.client import GoogleAdsClient
from google.ads.googleads.errors import GoogleAdsException

from datetime import datetime, timedelta
from typing import Optional, Dict
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [0]:
end_date = datetime.today().strftime("%Y-%m-%d")

start_date = (
    datetime.today() - timedelta(days=365)
).strftime("%Y-%m-%d")

logging.info(f"Reporting window: {start_date} to {end_date}")

2026-06-15 07:05:23,307 - INFO - Reporting window: 2025-06-15 to 2026-06-15


In [0]:
ga_service = client.get_service("GoogleAdsService")

In [0]:
query_map = {

    # 1. Campaign Performance
    "bronze_campaign_performance": f"""
SELECT
    segments.date,
    campaign.id,
    campaign.name,
    campaign.status,
    campaign.advertising_channel_type,
    campaign_budget.id,
    campaign_budget.name,
    campaign_budget.amount_micros,
    metrics.impressions,
    metrics.clicks,
    metrics.ctr,
    metrics.cost_micros,
    metrics.conversions
FROM campaign
WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
""",

    # 2. Keyword Performance
    "bronze_keyword_performance": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        ad_group.id,
        ad_group.name,
        ad_group_criterion.keyword.text,
        ad_group_criterion.keyword.match_type,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM keyword_view
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 3. Search Terms
    "bronze_search_terms": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        ad_group.id,
        ad_group.name,
        search_term_view.search_term,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM search_term_view
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

"bronze_geographic_performance": f"""
SELECT
    segments.date,
    campaign.id,
    campaign.name,
    geographic_view.country_criterion_id,
    geographic_view.location_type,
    metrics.impressions,
    metrics.clicks,
    metrics.conversions,
    metrics.cost_micros,
    metrics.ctr
FROM geographic_view
WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'""",

    # 5. Device Performance
    "bronze_device_performance": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        segments.device,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM campaign
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 6. Conversion Performance
    "bronze_conversion_performance": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        segments.conversion_action_name,
        metrics.conversions,
        metrics.conversions_value
    FROM campaign
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 7. Ad Group Performance
    "bronze_ad_group_performance": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        ad_group.id,
        ad_group.name,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM ad_group
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 8. Ad Performance
    "bronze_ad_performance": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        ad_group.id,
        ad_group.name,
        ad_group_ad.ad.id,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM ad_group_ad
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 9. Customer Information
    "bronze_customer_info": """
    SELECT
        customer.id,
        customer.descriptive_name,
        customer.currency_code,
        customer.time_zone
    FROM customer
    """,

    # 10. Campaign Details
    "bronze_campaign_details": """

        SELECT
            campaign.id,
            campaign.name,
            campaign.status,
            campaign.serving_status,
            campaign.advertising_channel_type,
            campaign.bidding_strategy_type,
            campaign_budget.id
        FROM campaign
        """,

    # 11. Budget Information
    "bronze_budget_information": """
    SELECT
        campaign_budget.amount_micros,
        campaign_budget.id,
        campaign_budget.name,
        campaign_budget.resource_name
    FROM campaign_budget
    """,

    # 12. Conversion Actions
    "bronze_conversion_actions": """
    SELECT
        conversion_action.id,
        conversion_action.name,
        conversion_action.type,
        conversion_action.status
    FROM conversion_action
    """,

    # 13. Age Performance
    "bronze_age_performance": f"""
    SELECT
        segments.date,
        age_range_view.resource_name,
        ad_group_criterion.age_range.type,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM age_range_view
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 14. Gender Performance
    "bronze_gender_performance": f"""
    SELECT
        segments.date,
        ad_group_criterion.gender.type,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM gender_view
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """,

    # 15. Landing Page Performance
    "bronze_landing_page_performance": f"""
    SELECT
        segments.date,
        campaign.id,
        campaign.name,
        landing_page_view.unexpanded_final_url,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions
    FROM landing_page_view
    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    """
}

In [0]:
from google.protobuf.json_format import MessageToDict
import pandas as pd

# Create Google Ads service
ga_service = client.get_service("GoogleAdsService")

# Customer ID
customer_id = "1401815809"

# Loop through all queries
for table_name, query in query_map.items():

    try:

        print(f"Loading {table_name}")

        # Execute GAQL query
        response = ga_service.search(
            customer_id=customer_id,
            query=query
        )

        records = []

        # Process each row in the response
        for row in response:

            # Convert protobuf row to dictionary
            row_dict = MessageToDict(
                row._pb,
                preserving_proto_field_name=True
            )

            # Flatten nested dictionaries
            flat_dict = pd.json_normalize(row_dict).to_dict("records")[0]

            records.append(flat_dict)

        # Create DataFrame and write to Delta table
        if records:

            df = spark.createDataFrame(records)

            (
                df.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(f"googleads_bronze.{table_name}")
            )

            print(f"{table_name} loaded successfully")

        else:

            print(f"No data found for {table_name}")

    except Exception as e:

        print(f"Error loading {table_name}: {e}")

Loading bronze_campaign_performance
bronze_campaign_performance loaded successfully
Loading bronze_keyword_performance
bronze_keyword_performance loaded successfully
Loading bronze_search_terms
bronze_search_terms loaded successfully
Loading bronze_geographic_performance
bronze_geographic_performance loaded successfully
Loading bronze_device_performance
bronze_device_performance loaded successfully
Loading bronze_conversion_performance
bronze_conversion_performance loaded successfully
Loading bronze_ad_group_performance
bronze_ad_group_performance loaded successfully
Loading bronze_ad_performance
bronze_ad_performance loaded successfully
Loading bronze_customer_info
bronze_customer_info loaded successfully
Loading bronze_campaign_details
bronze_campaign_details loaded successfully
Loading bronze_budget_information
bronze_budget_information loaded successfully
Loading bronze_conversion_actions
bronze_conversion_actions loaded successfully
Loading bronze_age_performance
bronze_age_perfor

In [0]:
s=spark.sql("SHOW TABLES IN googleads_bronze")
display(s)


database,tableName,isTemporary
googleads_bronze,bronze_ad_group_performance,false
googleads_bronze,bronze_ad_performance,false
googleads_bronze,bronze_age_performance,false
googleads_bronze,bronze_budget_information,false
googleads_bronze,bronze_campaign_details,false
googleads_bronze,bronze_campaign_performance,false
googleads_bronze,bronze_conversion_actions,false
googleads_bronze,bronze_conversion_performance,false
googleads_bronze,bronze_customer_info,false
googleads_bronze,bronze_device_performance,false


In [0]:
bro=spark.table("googleads_bronze.bronze_campaign_details")
display(bro)

campaign.advertising_channel_type,campaign.bidding_strategy_type,campaign.id,campaign.name,campaign.resource_name,campaign.serving_status,campaign.status,campaign_budget.id,campaign_budget.resource_name
PERFORMANCE_MAX,MAXIMIZE_CONVERSIONS,21519858184,Excel in Data Engineering,customers/1401815809/campaigns/21519858184,SERVING,PAUSED,13779178023,customers/1401815809/campaignBudgets/13779178023
DISPLAY,MAXIMIZE_CONVERSIONS,21771245709,Azure Data Engineering -Display - Image,customers/1401815809/campaigns/21771245709,ENDED,ENABLED,13969318520,customers/1401815809/campaignBudgets/13969318520
SEARCH,MAXIMIZE_CONVERSIONS,21781297084,Azure Data Engineering -Display - text,customers/1401815809/campaigns/21781297084,ENDED,PAUSED,13975409178,customers/1401815809/campaignBudgets/13975409178
SEARCH,MAXIMIZE_CONVERSIONS,22271705916,Leads-Search-24Feb2025,customers/1401815809/campaigns/22271705916,ENDED,PAUSED,14348313107,customers/1401815809/campaignBudgets/14348313107
SEARCH,MAXIMIZE_CONVERSIONS,22489991406,Hyderabad,customers/1401815809/campaigns/22489991406,ENDED,PAUSED,14516931095,customers/1401815809/campaignBudgets/14516931095
PERFORMANCE_MAX,MAXIMIZE_CONVERSIONS,22973632076,Data Engineering (AWS-Sept),customers/1401815809/campaigns/22973632076,SERVING,PAUSED,14898232496,customers/1401815809/campaignBudgets/14898232496
SEARCH,MAXIMIZE_CONVERSION_VALUE,22998241842,10-Sept AWS Snowflake,customers/1401815809/campaigns/22998241842,ENDED,PAUSED,14928117082,customers/1401815809/campaignBudgets/14928117082
SEARCH,MAXIMIZE_CONVERSIONS,23393627674,Azure-Data-Engineering-Jan-2026,customers/1401815809/campaigns/23393627674,SERVING,PAUSED,15234300935,customers/1401815809/campaignBudgets/15234300935
PERFORMANCE_MAX,MAXIMIZE_CONVERSIONS,23770266216,april 21st,customers/1401815809/campaigns/23770266216,SERVING,ENABLED,15516948212,customers/1401815809/campaignBudgets/15516948212
SEARCH,MAXIMIZE_CONVERSIONS,23770277166,APRIL 21ST,customers/1401815809/campaigns/23770277166,SERVING,ENABLED,15521655015,customers/1401815809/campaignBudgets/15521655015
